# Training a DSpark Draft Model for Qwen3-0.6B (online mode)

This notebook trains a **DSpark drafter** for **speculative decoding**, with
[`Qwen/Qwen3-0.6B`](https://huggingface.co/Qwen/Qwen3-0.6B) as the *verifier* (the model whose
output we want to reproduce exactly, only faster).

**How speculative decoding works.** Not every token is equally hard to predict. A closing bracket,
the rest of a common phrase, the second half of a familiar word - a tiny model guesses these just as
well as a big one. Speculative decoding exploits that imbalance with a **draft-then-verify** loop: a
small, fast *drafter* guesses a few tokens ahead, and the big model (the *verifier*, also called the
target model) checks all of those guesses in a **single** forward pass, keeping the ones it agrees
with and throwing away the rest.

Checking several drafted tokens at once costs about the same as generating one token, because
decoding is limited by memory bandwidth rather than arithmetic - that is where the speedup comes
from. And because every token still has to pass the verifier's own check, the text you get is
exactly what the verifier would have produced alone: this is a **lossless** speedup, not an
approximation. The number to watch is the **accepted length** - how many drafted tokens survive per
round. Raising it is the entire point of the training below.

**What DSpark does.** DSpark comes from
[*DSpark: Confidence-Scheduled Speculative Decoding with Semi-Autoregressive Generation*](https://arxiv.org/abs/2607.05147)
(arXiv:2607.05147). Two ideas:

- **It drafts a whole block at once.** Rather than producing candidates one at a time, DSpark's
  drafter emits `--block-size` tokens in a single pass. That is much faster, but a naive version
  makes every guess without seeing the guesses before it, so the later tokens in a block get
  rejected more often. DSpark bolts a small sequential module onto the parallel one (the paper calls
  the mix *semi-autoregressive*) so tokens in a block can build on each other, which keeps the tail
  of the block worth verifying.
- **It adapts how much it verifies.** When many users are served at once, DSpark shortens or
  lengthens the checked block per request instead of using one fixed size for everybody, so capacity
  is not spent on guesses that were likely to be discarded. This happens at inference time; the
  notebook only trains the drafter.

The paper reports 60-85% faster per-user generation than the MTP-1 baseline when deployed in the
DeepSeek-V4 serving system under live traffic.

**What the drafter consumes.** A DSpark drafter does not learn from text alone: it conditions on
the verifier's **auxiliary hidden states** (activations pulled from several layers of the verifier,
configured in step 3) alongside token embeddings. That is why a vLLM server has to run during
training.

**Where the hidden states come from.** The
[speculators](https://github.com/vllm-project/speculators) library offers three ways:

| Mode | Hidden states | Trade-off |
| --- | --- | --- |
| **online** (this notebook) | fetched from a live vLLM server on demand, discarded after use | no disk overhead, but the verifier occupies a GPU for the whole run |
| offline | pre-generated to disk before training starts | all GPUs go to generation first, then to training - but needs a lot of disk |
| hybrid | generated on demand during epoch 0, cached, reused afterwards | pays generation once, moderate disk |

**Hardware.** Tested on Kaggle with **2x T4**. Data generation uses both GPUs; during training
GPU 0 hosts the vLLM verifier server and GPU 1 trains the drafter.

**Sizing the run.** This notebook uses **1600 samples**, which is small enough to finish end to end
on 2x T4. You can turn that down freely - 200 to 400 samples is plenty to smoke-test the whole
pipeline in a fraction of the time. Two flags must move together: `--limit` in step 2a (how many
responses the verifier generates) and `--max-samples` in step 2b (how many get tokenized). Accepted
length scales with training data, so a drafter you actually intend to deploy wants far more than
1600 samples.

**Pipeline**

1. **Setup** - clone `speculators` and install it alongside vLLM.
2. **Prepare data** - regenerate responses with the verifier, then tokenize into speculator format.
3. **Serve the verifier** - start vLLM in hidden-state extraction mode.
4. **Train** - run `scripts/train.py` against that endpoint.
5. **Publish** - push the best checkpoint to the Hugging Face Hub.

Based on the official
[speculators training tutorial](https://docs.vllm.ai/projects/speculators/en/latest/user_guide/tutorials/train/).
For a friendly walkthrough of speculative decoding itself, see
[this article by Leonie Monigatti](https://leoniemonigatti.com/blog/speculative-decoding.html).

### Setup your environment

Clone the [speculators](https://github.com/vllm-project/speculators) repository and install it
together with vLLM. `speculators` provides the data-preparation and training scripts; vLLM both
serves the verifier and supplies the model-runtime code the trainer imports.

The upstream tutorial recommends two isolated virtual environments (one for speculators, one for
vLLM) to avoid dependency conflicts. On Kaggle a single environment is used here, with a couple of
manual fix-ups for pre-installed packages.

Clone the repo, then `cd` into it. **Every relative path used later** (`./scripts`,
`./magpie_Qwen3-0.6B.jsonl`, `./output`) is relative to this repository root - on Kaggle that is
`/kaggle/working/speculators`.

In [ ]:
!git clone https://github.com/vllm-project/speculators.git

In [ ]:
%cd speculators

Create the virtual environment and install the dependencies:

- `uv venv` / `uv pip install -e .` - installs `speculators` in editable mode so the `scripts/`
  entry points resolve against the cloned source.
- `vllm>=0.22.0` - the first release with the `extract_hidden_states` speculative method used in
  step 3.

> **Note:** `source speculators_venv/bin/activate` inside a `!` cell only affects that one
> subshell - it does **not** persist to later cells. The cells below therefore run against the
> notebook's own interpreter, which is why the packages are installed there too.

In [ ]:
# Speculators venv (for data prep and training)
! uv venv speculators_venv
! source speculators_venv/bin/activate
! uv pip install "vllm>=0.22.0"
! uv pip install -e .

Two Kaggle-specific fix-ups:

- **`pip uninstall torchaudio`** - the pre-installed `torchaudio` is pinned to Kaggle's `torch`
  build and breaks the import chain once vLLM upgrades `torch`. Nothing here needs audio.
- **`pip install -Uq datasets`** - the data-prep step needs a newer `datasets` than the image
  ships. `--break-system-packages` is required because the image's site-packages is managed.

In [ ]:
! pip uninstall -y torchaudio
! pip install -Uq datasets --break-system-packages

### Prepare Your Data

The drafter is trained to imitate **this** verifier's output distribution, so the training targets
must be text that `Qwen3-0.6B` actually produced - not the original human- or GPT-written responses
in whatever dataset you start from.

Two steps:

1. **Response regeneration** - replay a prompt set through the verifier and keep its own answers.
2. **`prepare_data.py`** - apply the serving chat template, tokenize, and derive a `loss_mask`
   (so loss is computed on assistant tokens only), writing the result in *speculator format*.

Regenerate responses for the [magpie](https://huggingface.co/datasets/Magpie-Align) prompt set
using the verifier itself.

| flag | meaning |
| --- | --- |
| `--model` | the verifier that generates the responses |
| `--dataset magpie` | which built-in prompt source to replay |
| `--limit 1600` | **how many samples to generate - the main cost knob.** Drop it to ~200-400 to smoke-test the pipeline quickly; raise it well above 1600 for a drafter you plan to deploy |
| `--gpus 0,1` + `--dp-size 2` | data-parallel generation across both T4s |

Output: **`./magpie_Qwen3-0.6B.jsonl`**. This is the longest-running cell in the notebook and its
runtime is roughly linear in `--limit`, so this is the first thing to lower if the run has to fit a
shorter session. Whatever you choose here, match it with `--max-samples` in the next cell.

In [ ]:
! ./scripts/response_regeneration/run_all.sh \
  --model "Qwen/Qwen3-0.6B" \
  --dataset magpie \
  --limit 1600 \
  --gpus 0,1 \
  --dp-size 2

Convert the raw JSONL conversations into the tokenized *speculator format* the trainer expects
(`input_ids` + `loss_mask` per sample).

| flag | meaning |
| --- | --- |
| `--model` | tokenizer and chat template to use (must match the verifier) |
| `--data` | the JSONL produced above |
| `--output ./output` | destination for the processed dataset |
| `--max-samples 1600` | how many samples to tokenize - **lower this in step with `--limit` above.** It can only cap what was already generated, so setting it higher than `--limit` adds nothing |
| `--seq-length 2048` | truncation length in tokens |
| `--overwrite` | makes re-running this cell idempotent |

`./output` ends up holding Arrow dataset shards plus **token-frequency statistics**. Those
statistics are what makes the reduced `--draft-vocab-size` in step 4 possible: the drafter only
needs the tokens the verifier actually emits often.

In [ ]:
# in speculators venv
! python scripts/prepare_data.py \
  --model "Qwen/Qwen3-0.6B" \
  --data ./magpie_Qwen3-0.6B.jsonl \
  --output ./output \
  --max-samples 1600 \
  --seq-length 2048 \
  --overwrite

### **Start a vLLM server to get Hidden States**

DSpark conditions on the verifier's intermediate activations, so training needs hidden states for
every token in the dataset. In **online** mode nothing is precomputed - a vLLM server runs
alongside training and hands them over on request.

This server must stay up for the entire training run.

Launch vLLM in **hidden-state extraction** mode. The two config blobs are what make this work:

- **`speculative_config`**
  - `"method": "extract_hidden_states"` - vLLM is not doing speculative decoding here; this mode
    tells it to *export* activations instead.
  - `eagle_aux_hidden_state_layer_ids: [2, 14, 25, 28]` - which auxiliary layers to export.
    These sample early / mid / late points of Qwen3-0.6B's 28-layer stack, giving the drafter a
    multi-scale view of the verifier's internal state.
- **`kv_transfer_config`** - registers `ExampleHiddenStatesConnector` as a `kv_producer` that
  writes the extracted tensors under `shared_storage_path` (`/tmp/hidden_states`), where the
  trainer picks them up.

The server is started with `subprocess.Popen(..., start_new_session=True)` so it lands in its own
process group: interrupting a notebook cell (SIGINT) will **not** kill it. Logs are redirected to
`/content/vllm.log`, and the server listens on the vLLM default, port **8000**.

In [ ]:
import subprocess, json

speculative_config = {
    "method": "extract_hidden_states",
    "num_speculative_tokens": 1,
    "draft_model_config": {
        "hf_config": {"eagle_aux_hidden_state_layer_ids": [2, 14, 25, 28]}
    }
}

kv_transfer_config = {
    "kv_connector": "ExampleHiddenStatesConnector",
    "kv_role": "kv_producer",
    "kv_connector_extra_config": {"shared_storage_path": "/tmp/hidden_states"}
}

log_file = open("/content/vllm.log", "w")
process = subprocess.Popen(
    [
        "vllm", "serve", "Qwen/Qwen3-0.6B",
        "--speculative_config", json.dumps(speculative_config),
        "--kv_transfer_config", json.dumps(kv_transfer_config),
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    start_new_session=True,  # detaches from the notebook's process group so a cell interrupt (SIGINT) doesn't reach it
)
print(f"Started vLLM server with PID {process.pid}")

Poll `/health` until the server answers - up to 60 attempts, 5 s apart (~5 minutes).

The first start is slow: it downloads the model, loads weights, and captures CUDA graphs. If it
never reports ready, read the log (`!tail -n 50 /content/vllm.log`) - out-of-memory errors and
port conflicts both show up there.

In [ ]:
import time, requests

for i in range(60):
    try:
        r = requests.get("http://127.0.0.1:8000/health")
        if r.status_code == 200:
            print("Server is ready!")
            break
    except requests.exceptions.ConnectionError:
        pass
    print(f"Waiting... ({i+1})")
    time.sleep(5)

### **Train DSpark draft model**

Now train the drafter. For each batch the trainer pulls the verifier's hidden states from the vLLM
endpoint, runs the DSpark draft head on them, and updates the drafter to predict what the verifier
would generate next.

Checkpoints are written to `./output/checkpoints`, with the best-scoring one kept at
`./output/checkpoints/checkpoint_best` - that is what gets published in the final step.

`CUDA_VISIBLE_DEVICES=1` matters here: GPU 0 is busy hosting the vLLM verifier, so training is
confined to GPU 1 (`torchrun --nproc_per_node 1`, i.e. a single process). `tee /content/train.log`
keeps a copy of the output.

| flag | meaning |
| --- | --- |
| `--verifier-name-or-path` | the model the drafter learns to imitate |
| `--data-path` / `--save-path` | prepared dataset in / checkpoints out |
| `--draft-vocab-size 32000` | reduced drafter vocabulary (derived from the token-frequency stats) - a smaller output head means a cheaper draft step |
| `--epochs 5`, `--lr 3e-4` | optimization settings |
| `--total-seq-len 1024` | maximum training sequence length |
| `--max-anchors 128` | cap on anchor positions sampled per sequence for the DSpark objective |
| `--speculator-type dspark` | draft architecture (alternatives: `eagle3`, `peagle`, `dflash`, `mtp`) |
| `--num-layers 5` | depth of the draft model |
| `--block-size 4` | tokens the drafter proposes per verification step |
| `--loss-fn '{"ce": 0.1, "tv": 0.9}'` | blended loss, weighted mostly toward the `tv` term, which pulls the drafter's predictions toward the verifier's - the closer they match, the more tokens get accepted |
| `--vllm-endpoint` | where to fetch hidden states from |
| `--on-missing generate` | hidden state not available - ask the server for it (vs. `raise` / `skip`) |
| `--on-generate delete` | discard it after use; this is what makes the run *online* |
| `--no-resume-from-checkpoint` | start fresh rather than continuing a previous run |
| `--log-freq 1` | log every step |

In [ ]:
# in speculators venv
! CUDA_VISIBLE_DEVICES=1 torchrun --standalone --nproc_per_node 1 \
  scripts/train.py \
  --verifier-name-or-path "Qwen/Qwen3-0.6B" \
  --data-path ./output \
  --save-path ./output/checkpoints \
  --draft-vocab-size 32000 \
  --epochs 5 \
  --total-seq-len 1024 \
  --max-anchors 128 \
  --num-workers 2 \
  --speculator-type dspark \
  --num-layers 5 \
  --block-size 4 \
  --lr 3e-4 \
  --loss-fn '{"ce": 0.1, "tv": 0.9}' \
  --vllm-endpoint http://localhost:8000/v1 \
  --on-missing generate \
  --on-generate delete \
  --no-resume-from-checkpoint \
  --log-freq 1 2>&1 | tee /content/train.log

Shut the verifier down. `os.killpg` targets the whole process group created by
`start_new_session=True`, so the vLLM workers go with it - a plain `process.kill()` would leave
children holding GPU memory.

Run this once training finishes (or before re-running the server cell) to free GPU 0 and port 8000.

In [ ]:
import os, signal
os.killpg(os.getpgid(process.pid), signal.SIGTERM)

### Save best draft model to huggingface

Publish `checkpoint_best` - the drafter with the best validation score - to the Hugging Face Hub so
it can be loaded as a speculator by vLLM later.

Load the Hugging Face **write** token from Kaggle Secrets and expose it as `HF_TOKEN`, which
`huggingface_hub` picks up automatically.

Replace `HF_WRITE_YOSEFW` with the name of your own secret: create a Hugging Face **write** token
on the Hub, attach it to this notebook under Add-ons -> Secrets, and use that name here. Outside
Kaggle, set `HF_TOKEN` another way - e.g. `huggingface_hub.login()` or an environment variable.

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_WRITE_YOSEFW")

Create the target repo (`exist_ok=True` makes re-runs safe) and upload the checkpoint folder.

Replace `yosefw/` in `repo_id` with your own Hugging Face username - the write token set above
can only push to repos under your own account.

`folder_path` is the absolute Kaggle path to `checkpoint_best`; adjust it if you cloned
`speculators` somewhere else.

**Next step:** point vLLM at the published repo as the speculative model and measure the acceptance
rate - that is the number that tells you whether the drafter is earning its keep.

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
api.create_repo(repo_id="yosefw/Qwen3-0.6B-DSpark", exist_ok=True)

api.upload_folder(
    folder_path="/kaggle/working/speculators/output/checkpoints/checkpoint_best",
    repo_id="yosefw/Qwen3-0.6B-DSpark",
    repo_type="model",
)